# Chat Completions API

In [ ]:
# 공통 설정
# .env 파일에서 API Key와 기본 모델명을 읽어온다.
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPEAI_API_KEY를 환경 변수로 설정하세요.")

client = OpenAI(api_key=api_key)
DEFAULT_MODEL = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-4.1-mini")

print("OpenAI client 준비 완료")
print("기본 모델 : ", DEFAULT_MODEL)

OpenAI client 준비 완료
기본 모델 :  gpt-4.1-mini


## 기본 호출
- system 메세지는 모델의 역할과 답변 규칙을 지정한다.
- user 메세지는 사용자의 실제 요청이다.

In [ ]:
response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[
        {"role": "system", "content" : "너는 초급 개발자에게 쉽게 설명하는 AI 강사이다."},
        {"role": "user", "content": "Chat Completions API의 messages 구조를 설명해줘."},
    ],
    temperature=0.3
)

print(response.choices[0].message.content)

네! Chat Completions API의 `messages` 구조를 쉽게 설명해줄게요.

`messages`는 대화 내용을 담는 배열이에요. 이 배열 안에는 여러 개의 메시지가 들어가고, 각 메시지는 누가 말했는지와 그 내용이 포함되어 있어요.

각 메시지는 보통 이렇게 생겼어요:

```json
{
  "role": "사용자" 또는 "시스템" 또는 "어시스턴트",
  "content": "메시지 내용"
}
```

- **role**: 누가 말했는지를 나타내요.
  - `"system"`: 대화의 설정이나 지침을 알려줄 때 사용해요. 예를 들어, "친절하게 답변해줘" 같은 역할이에요.
  - `"user"`: 사용자가 보낸 메시지에요. 우리가 질문하거나 요청하는 내용이죠.
  - `"assistant"`: AI가 답변한 메시지에요.

- **content**: 실제 대화 내용, 즉 메시지 텍스트가 들어가요.

예를 들어, 이런 식으로 쓸 수 있어요:

```json
[
  {"role": "system", "content": "친절하고 간단하게 답변해줘."},
  {"role": "user", "content": "Chat Completions API가 뭐야?"},
  {"role": "assistant", "content": "Chat Completions API는 대화형 AI를 만들 때 사용하는 API예요."}
]
```

이렇게 `messages` 배열에 대화 내용을 차례대로 넣으면, AI가 그 흐름을 이해하고 적절한 답변을 만들어줘요.

더 궁금한 점 있으면 언제든 물어봐요!


## 대화 이력 직접 누적하기
Chat Completions API에서는 이전 대화를 API가 자동으로 기억하지 않는다.
따라서 이어지는 대화를 만드려면 message 리스트에 사용자 질문과 모델 답변을 직접 누적해야 한다.

In [ ]:
messages = [
    {"role" : "system", "content": "너는 Python 수업을 돕는 AI 튜터이다. 답변은 3문장 이내로 한다."}
]

def chat(user_input):
    # 사용자의 새 질문을 대화 이력에 추가
    messages.append({"role" : "user", "content": user_input})

    # 지금까지의 전체 대화 이력을 모델에 전달
    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=messages,
        temperature=0.4
    ) 

    # 모델 답변을 꺼내고 다음 턴을 위해 다시 이력에 추가
    answer = response.choices[0].message.content
    messages.append({"role" : "assistant", "content": answer})
    return answer

print(chat("함수와 메서드의 차이를 설명해줘."))
print()
print(chat("방금 설명을 Java 예시로 바꿔줘."))

함수는 독립적으로 정의되어 호출되는 코드 블록이고, 메서드는 특정 객체에 속한 함수입니다. 메서드는 객체의 상태를 다루거나 객체와 관련된 동작을 수행합니다. 즉, 함수는 일반적인 동작, 메서드는 객체 지향 프로그래밍에서 객체와 연관된 동작입니다.

Java에서 함수는 클래스 밖에 없고, 메서드는 클래스 내부에 정의된 함수입니다. 예를 들어, `public static void print()`는 함수처럼 사용할 수 있고, `public void show()`는 객체의 메서드입니다. 메서드는 객체의 상태를 변경하거나 참조할 수 있습니다.


## system 메세지 변경을 통해 답변 스타일 변경

In [ ]:
system_messages = [
    "너는 초급 개발자 대상 강사다. 쉬운 용어와 간단한 예시로 설명한다.",
    "너는 백엔드 실무자에게 설명하는 멘토다. 실무 예시를 포함해서 설명한다.",
    "너는 백엔드 개발자 면접관이다. 핵심 개념과 실무 판단 기준을 중심으로 설명한다."
]

user_question = "API에서 토큰 사용량을 로깅해야 하는 이유를 설명해줘."

for idx, system_message in enumerate(system_messages, start=1):
    practice_messages = [
        {"role":"system", "content":system_message},
        {"role":"user", "content":user_question},
    ]

    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=practice_messages,
        temperature=0.3
    )

    print(f"[{idx}] {system_message}")
    print(response.choices[0].message.content)
    print("-" * 60)


[1] 너는 초급 개발자 대상 강사다. 쉬운 용어와 간단한 예시로 설명한다.
좋아! 아주 쉽게 설명해줄게.

API에서 **토큰 사용량을 로깅**한다는 건, API를 쓸 때 얼마나 많은 '토큰'을 썼는지 기록하는 거야.

여기서 '토큰'은 API가 처리하는 데이터의 작은 조각 같은 거야. 예를 들어, 문장을 단어로 나누면 그 단어들이 토큰이 될 수 있어.

왜 토큰 사용량을 기록해야 할까?

1. **비용 관리**  
   API는 보통 토큰 사용량에 따라 비용이 달라져. 많이 쓰면 돈도 많이 내야 해. 그래서 얼마나 썼는지 기록해서 비용을 잘 관리해야 해.

2. **사용량 모니터링**  
   누가 얼마나 많이 쓰는지 알 수 있어. 너무 많이 쓰면 알림을 주거나 제한을 걸 수 있어.

3. **문제 해결**  
   만약 API가 느려지거나 오류가 생기면, 토큰 사용량 기록을 보면 문제 원인을 찾기 쉬워.

4. **통계와 분석**  
   어떤 기능이 많이 쓰이는지, 사용자들이 어떻게 API를 사용하는지 알 수 있어. 그래서 더 좋은 서비스를 만들 수 있어.

예를 들어, 네가 카페에서 커피를 팔 때, 하루에 몇 잔 팔렸는지 기록하면 돈 관리도 쉽고, 인기 메뉴도 알 수 있잖아? API 토큰 사용량 기록도 그런 거야!

이해됐지? 더 궁금한 거 있으면 물어봐!
------------------------------------------------------------
[2] 너는 백엔드 실무자에게 설명하는 멘토다. 실무 예시를 포함해서 설명한다.
API에서 토큰 사용량을 로깅해야 하는 이유는 여러 가지가 있습니다. 실무 예시와 함께 설명해드릴게요.

1. **비용 관리 및 최적화**  
   많은 API 서비스, 특히 AI나 클라우드 기반 API는 토큰 사용량에 따라 과금됩니다.  
   - 예시: OpenAI API를 사용한다고 가정해봅시다. 요청마다 소비하는 토큰 수에 따라 비용이 발생합니다. 토큰 사용량을 로깅하지 않으면, 어느 기능이나 요청이 비용